In [ ]:
#| hide
%load_ext autoreload
%autoreload 2

# InstructorParser

> Using OpenAI and instructor to parse the affiliation information

In [ ]:
#| default_exp instructor_parser

In [ ]:
#| hide
from nbdev.showdoc import *
from dotenv import load_dotenv

In [ ]:
#| hide
load_dotenv()

True

In [ ]:
#| export
import os

In [ ]:
#| export
from pydantic import BaseModel, model_validator, Field, field_validator, EmailStr
from typing import Any
import openai

In [ ]:
#| export
class Affiliation(BaseModel):
    """Affiliation data representation"""
    organization:str | None = Field(
        default= "",
        description= "The company or university of the Affiliation",        
    )
    laboratory: str | None = Field(
        default= "",
        description= "The Laboratory of the Affiliation",        
    )
    department: str | None =Field(
        default= "",
        description= "The department of the Affiliation",        
    )
    faculty: str | None =Field(
        default= "",
        description= "The faculty of the Affiliation",        
    )
    country: str | None = Field(
        default= "",
        description= "The country of the Affiliation",        
    )
    city: str | None = Field(
        default= "",
        description= "The city of the Affiliation",        
    )
    state: str | None = Field(
        default= "",
        description= "The state, county, or province of the Affiliation",        
    )
    email: EmailStr | None= Field(
        default= None,
        description= "The email Address",        
    )
    

In [ ]:
#| export
class InstructorParser(BaseModel):
    """Class to activate the OpenAI Parser algorithm"""
    data_class: Any
    model: str = 'gpt-4-1106-preview'

    @model_validator(mode='before')
    @classmethod
    def validate_env(cls, values):
        try:
            import instructor
            cls.client = instructor.patch(openai.OpenAI())
        except ImportError:
            raise ImportError(
                "Could not import instructor python package. "
                "This is needed in order to accurately extract the data "
                "from Affiliations. Please install it with `pip install instructor`."
            )

        if not values['data_class']:
            raise Exception("It is needed a data class to guide the LLM to extract information")
        return values
    
    def run(self, text):
        data: self.data_class = self.client.chat.completions.create(
            model=self.model,
            response_model=self.data_class,
            messages=[{
                "role": "system", 
                "content": "Your role is to extract data from the following text chunk."
            }, {
                "role": "user", 
                "content": text
            }])
        return data

    def to_dict(self):
        return self.data_class.model_dump()

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()